In [1]:
# Run once if needed
# !pip install langchain langchain-ollama langchain-core pydantic

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field, ValidationError
from typing import List
import json

Define the Pydantic schema

In [2]:
class DocumentMetadata(BaseModel):
    title: str = Field(..., description="Title of the document")
    author: str = Field(..., description="Author of the document")
    publication_date: str = Field(..., description="Date in YYYY-MM-DD format")
    keywords: List[str] = Field(..., description="List of keywords")
    document_type: str = Field(..., description="Type of the document, e.g., report, article")

Model

In [7]:
# Change the model name to whatever you have pulled
model = ChatOllama(
    model="llama3.2",          # or "llama3.2", "mistral", "qwen2.5", etc.
    temperature=0,             # deterministic for extraction
    # base_url="http://localhost:11434"  # only needed if not default
)

In [8]:
structured_model = model.with_structured_output(DocumentMetadata)

Sample unstructured documents

In [9]:
documents = [
    """This report, 'AI Trends 2025', was written by Dr. Sarah Lee and published on 2025-05-01.
       It covers topics like artificial intelligence, deep learning, and ethics in AI.""",
    
    """The article 'Climate Change and Agriculture' by John Smith was released on 2024-09-15.
       It discusses sustainability, farming practices, and environmental policy.""",
    
    # Add more raw texts here
]

Extract + validate (using with_structured_output)

In [10]:
structured_metadata = []

for i, doc in enumerate(documents, 1):
    print(f"Processing document {i}...")
    try:
        # The model returns a validated Pydantic object directly
        metadata = structured_model.invoke(
            f"Extract the metadata from this document text:\n\n{doc}"
        )
        structured_metadata.append(metadata.model_dump())   # → dict
        print("  Success:", metadata.model_dump())
    except Exception as e:
        print(f"  Failed: {e}")
        # You can decide to skip or flag the document

Processing document 1...
  Success: {'title': 'AI Trends 2025', 'author': 'Dr. Sarah Lee', 'publication_date': '2025-05-01', 'keywords': ['artificial intelligence', 'deep learning', 'ethics in AI'], 'document_type': 'report'}
Processing document 2...
  Success: {'title': 'Climate Change and Agriculture', 'author': 'John Smith', 'publication_date': '2024-09-15', 'keywords': ['sustainability', 'farming practices', 'environmental policy'], 'document_type': 'article'}


Alternative classic way (Prompt + JsonOutputParser)

In [11]:
parser = JsonOutputParser(pydantic_object=DocumentMetadata)
# or: parser = PydanticOutputParser(pydantic_object=DocumentMetadata)

prompt = PromptTemplate(
    template="""Extract the following metadata from the document text:
title, author, publication_date (YYYY-MM-DD), keywords, document_type.

Return ONLY valid JSON that matches this schema:
{format_instructions}

Document text:
{document_text}
""",
    input_variables=["document_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

In [12]:
chain = prompt | model | parser

structured_metadata = []
for doc in documents:
    try:
        result = chain.invoke({"document_text": doc})
        # result is already a dict (or Pydantic if you used PydanticOutputParser)
        if hasattr(result, "model_dump"):
            structured_metadata.append(result.model_dump())
        else:
            structured_metadata.append(result)
    except Exception as e:
        print("Parsing failed:", e)

In [13]:
print("\n=== Final structured dataset ===")
for idx, data in enumerate(structured_metadata, 1):
    print(f"\nDocument {idx}:")
    print(json.dumps(data, indent=2))


=== Final structured dataset ===

Document 1:
{
  "title": "AI Trends 2025",
  "author": "Dr. Sarah Lee",
  "publication_date": "2025-05-01",
  "keywords": [
    "artificial intelligence",
    "deep learning",
    "ethics in AI"
  ],
  "document_type": "report"
}

Document 2:
{
  "title": "Climate Change and Agriculture",
  "author": "John Smith",
  "publication_date": "2024-09-15",
  "keywords": [
    "sustainability",
    "farming practices",
    "environmental policy"
  ],
  "document_type": "article"
}
